In [1]:
import omero
import ezomero

import omero.scripts as scripts
from omero.gateway import BlitzGateway
import omero
import tqdm
import pandas as pd

from omero.rtypes import rint, rlong, rstring, robject, unwrap

In [2]:
host = 'omero-int.biotec.tu-dresden.de'
user = 'johamuel'  # replace this with your username
secure = True
port = 4064
group = 'Dye Lab'

conn = ezomero.connect(host=host, user=user, secure=secure, port=port, group=group)

In [35]:
screen_id = 3

In [40]:
dataset_ids = ezomero.get_dataset_ids(conn, project=145)
for dataset_id in dataset_ids:
    image_ids = ezomero.get_image_ids(conn, dataset=dataset_id)

    plate = omero.model.PlateI()
    plate.name = omero.rtypes.RStringI(conn.getObject('Dataset', dataset_id).name)

    plate.columnNamingConvention = rstring("letter")
    # 'letter' or 'number'
    plate.rowNamingConvention = rstring("number")

    update_service = conn.getUpdateService()
    plate = update_service.saveAndReturnObject(plate)

    link = omero.model.ScreenPlateLinkI()
    link.parent = omero.model.ScreenI(screen_id, False)
    link.child = omero.model.PlateI(plate.id.val, False)
    update_service.saveObject(link)

    df = pd.DataFrame()
    for image_id in tqdm.tqdm(image_ids):
        map_annotation_ids = ezomero.get_map_annotation_ids(conn, object_type='Image', object_id=image_id)
        kv_pairs = ezomero.get_map_annotation(conn, map_ann_id=map_annotation_ids[0])
        kv_pairs['image_id'] = image_id
        df = pd.concat([df, pd.DataFrame(kv_pairs, index=[0])])

    df["row"] = df["well"].apply(lambda x: int(x[1:]))
    df["column"] = df["well"].apply(lambda x: ord(x[0]) - 64 - 1)
    _df = df.groupby(["row", "column"]).agg({"row": 'first', 'column': 'first', 'image_id': lambda x: list(x)}).reset_index(drop=True)

    for idx, row in _df.iterrows():
        well = omero.model.WellI()

        well = omero.model.WellI()
        well.plate = omero.model.PlateI(plate.getId().getValue(), False)
        well.column = rint(row.column)
        well.row = rint(row.row)

        for image_id in row.image_id:
            ws = omero.model.WellSampleI()
            ws.image = omero.model.ImageI(image_id, False)
            ws.well = well
            well.addWellSample(ws)
        update_service.saveObject(well)

  0%|          | 1/242 [00:00<02:48,  1.43it/s]


KeyboardInterrupt: 

In [38]:
kv_pairs

{'date': '20220502',
 'pdo_line': 'OO116cN',
 'well': 'D04',
 'image_index': 'F005',
 'image_id': 34976}

In [33]:
for idx, row in _df.iterrows():
    well = omero.model.WellI()

    well = omero.model.WellI()
    well.plate = omero.model.PlateI(plate.getId().getValue(), False)
    well.column = rint(row.column)
    well.row = rint(row.row)

    for image_id in row.image_id:
        ws = omero.model.WellSampleI()
        ws.image = omero.model.ImageI(image_id, False)
        ws.well = well
        well.addWellSample(ws)
    update_service.saveObject(well)

In [25]:
row.row

3